# Ron — فحص بيئة Colab واختبار تدريب صغير

هذا الدفتر لا يغيّر أوزان رون الأصلية ولا يشغّل التدريب الطويل. يفحص موارد الجلسة، يختبر تثبيت المشروع والاختبارات الأساسية، ثم ينفّذ 3 خطوات تدريب على نموذج Ron صغير اصطناعيًا ويحفظ ملف اختبار منفصلًا.

افتح الدفتر في Google Colab ثم اختر **Runtime → Run all**. إذا انتهى الاختبار بنجاح، ننتقل لتجهيز تشغيل التدريب الفعلي وحفظ الأوزان في Google Drive.

In [ ]:
import os, sys, platform, subprocess, json, time
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print('GPU not assigned; smoke test will use CPU.')
except Exception as e:
    print('PyTorch import failed:', repr(e))
    raise
print('Free disk (GiB):', round(__import__('shutil').disk_usage('/content').free / 1024**3, 2))

In [ ]:
REPO = '/content/Ron-0'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','feat/native-core-audit','https://github.com/arkanws513-spec/Ron-0.git', REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin','feat/native-core-audit'], check=True)
    subprocess.run(['git','-C',REPO,'checkout','feat/native-core-audit'], check=True)
os.chdir(REPO)
print(subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-e','.[model]','pytest'], check=True)

In [ ]:
# Run focused regression tests before any training.
subprocess.run([sys.executable,'-m','pytest','tests/test_model.py','tests/test_native_10m_recipe.py','tests/test_sft_10m.py','tests/test_native_evaluation_gate.py','-q'], check=True)

In [ ]:
# Tiny isolated smoke test: small random data, 3 optimizer updates, separate output file.
import torch
from ron.model import RonCausalLM
from ron.model_config import RonModelConfig
torch.manual_seed(1701)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = RonModelConfig(vocab_size=128, hidden_size=64, num_layers=2, num_heads=4, max_sequence_length=32, dropout=0.0)
model = RonCausalLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
parameter_count = sum(p.numel() for p in model.parameters())
losses = []
model.train()
for step in range(3):
    x = torch.randint(0, cfg.vocab_size, (2, 32), device=device)
    y = torch.randint(0, cfg.vocab_size, (2, 32), device=device)
    optimizer.zero_grad(set_to_none=True)
    out = model(x, y)
    assert out.loss is not None and torch.isfinite(out.loss), 'loss is not finite'
    out.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(float(out.loss.detach().cpu()))
smoke_path = '/content/ron_smoke_test_only.pt'
torch.save({'config': cfg.__dict__, 'model_state_dict': model.state_dict(), 'smoke_test_only': True, 'losses': losses}, smoke_path)
print(json.dumps({'status':'PASS','device':str(device),'parameters':parameter_count,'steps':len(losses),'losses':losses,'test_checkpoint':smoke_path,'original_checkpoint_modified':False}, ensure_ascii=False, indent=2))
assert os.path.isfile(smoke_path) and os.path.getsize(smoke_path) > 0

## ما الذي يعنيه النجاح؟

نجاح هذا الدفتر يثبت فقط أن البيئة قادرة على استيراد نواة رون وتنفيذ تمرير أمامي/خلفي وحفظ ملف اختبار صغير. **لا يعني أن رون أصبح أذكى أو أن نموذج 10M تدرب.** بعده نراجع سجل الاختبارات والموارد المتاحة ونبدأ تجربة قصيرة على بيانات المشروع، مع حفظ نقطة استكمال مستقلة وعدم استبدال الملف الحي قبل اجتياز التقييم.